# report11 — 검출기 교정 ②: 저속 표적·분해능

**핵심.** 저속 표적·거리 분해능·위치 관측성은 신호 대역폭과 송수신 기하가 정하는 **원리적 문턱**이다 — 검출 사슬 자체는 표준(pyAPRiL 로 교차검증)이고, 표적 밝기 σ 만 SBR+PO 로 넣으면 그 문턱 위에서 드론이 실제로 잡힌다.

| 이 리포트의 척추 |  |
|---|---|
| **① Sionna 의 공백** | 바이스태틱 거리 분해능은 **ΔR_b = c/B**(B=기준신호 대역폭; 모노스태틱 등가는 c/2B), 저속 맹점과 위치 관측성은 **송수신 기하**로 못박혀 있어 시뮬레이터가 바꿀 수 없다. 게다가 스톡 Sionna 는 파형·전파만 줄 뿐 이 문턱을 재는 **검출 신호처리(ECA·CAF·CFAR)가 아예 없다**. |
| **② 선행 연구의 방식** | 패시브 레이더 검출은 ECA→CAF→CFAR 표준 사슬이며 하드웨어 선행이 그대로 썼다 — 5G NR OFDM 패시브 레이더(Wypich·Zielinski, Sensors 2026, DOI 10.3390/s26041317, ECA⁺→CA-CFAR) · LTE450 드론 패시브 레이더(Demissie, IET RSN 2025, DOI 10.1049/rsn2.70092). 5G SSB 상시 신호의 거리·속도 분해능/CRB 한계는 Jopanya·Osorio(SPAWC 2025, arXiv:2504.02641)가 분석했다. |
| **③ 쓴 라이브러리·결합** | 검출 사슬은 새로 짜지 않고 오픈소스 **pyAPRiL**(GPLv3, BME Radarlab)의 `cc_detector`(CAF)·`CA_CFAR` 를 그대로 쓴다(중복 구현 회피). 표적 밝기 σ 만 자작 SBR+PO 로 계산해 Sionna 채널에 주입한다 — 파형·전파는 Sionna PHY·RT. |
| **④ 검증** | pyAPRiL 로 교차검증 — NR·WiFi·LTE **3/3 모드 모두** 정답 거리빈(오차 0 빈)에 검출(verify_pyapril.json). 분해능은 이론 ΔR_b=c/B 대비 0.89~0.94배(sinc 0.886), 링크버짓 3독립계산 최대 편차 3e-14 dB, 관측성 회전대칭은 기계정밀도까지 0. |

---


## 📋 이 결과가 어디서 어떻게 나왔나

> 이 절은 **직접 참여하지 않은 사람도 출처를 따라가고 재현할 수 있도록** 넣었습니다. 버전·GPU 는 노트북 생성 시점에 **실제로 읽어온 값**입니다.

### 1️⃣ 무엇을 참고했나

| 항목 | 출처 | 성격 |
|---|---|---|
| ECA 노치·블라인드 속도(운용 적분시간) | outputs/verify_eca.json | 측정 (RT 클러터 + ECA 사영) |
| 모호함수·거리 분해능 측정/이론 | outputs/verify_ambiguity.json | 측정 (기준신호 자기모호함수) |
| 레이더방정식·처리이득·잡음바닥·SCR·Pd | outputs/verify_linkbudget.json · outputs/verify_cfar.json(roc_NR100) | 측정 (σ 는 SBR, 3GPP TS / ITU-R P.2040 재질) |

### 2️⃣ 어떤 도구가 무엇을 했나 — **Sionna 내부인가, 우리가 짠 건가**

| 도구 | 하는 일 | 어디서 도는가 |
|---|---|---|
| `radar-dsp` | 레이더 신호처리 (`src/passive_process.py`) — ECA(직접파 제거) · 거리-도플러 · CA-CFAR | 🔴 **별도** (numpy, CPU). **Sionna 에 레이더 DSP 는 없다** |
| `sbr` | SBR (`src/rcs_sbr.py`) — **Mitsuba 광선 + PO 표면적분**으로 RCS. 가림(occlusion) 포함 | 🟡 **우리가 짰다** — 다만 광선추적은 Sionna 가 쓰는 **Mitsuba 3 엔진 그대로** (GPU). Sionna 에 RCS 솔버가 없기 때문 |
| `sionna-phy` | Sionna PHY (`ofdm`/`nr`/`channel`) — OFDM 변복조 · 3GPP 뉴머롤로지 · RT 경로를 신호에 적용 | 🟢 **Sionna 내부** (PyTorch 백엔드, GPU) |
| `matplotlib` | matplotlib — 도표·그래프 | 🔴 **별도** (CPU). 계산 결과를 *그리기만* 한다 |

> 🔑 **이 구분이 이 프로젝트에서 가장 자주 오해받는 지점입니다.**
> - **전파**(경로·지연·도플러·렌더·라디오맵)는 🟢 **Sionna 가** 합니다.
> - **표적 RCS** 는 🟡 우리가 짠 **SBR** 이 합니다 — Sionna 에 RCS 솔버가 없기 때문입니다. 다만 광선추적은 Sionna 가 쓰는 **Mitsuba 3 엔진을 그대로** 씁니다.
> - **레이더 신호처리**(ECA/CFAR)는 🔴 우리가 짰습니다 — Sionna 에 레이더 DSP 가 없습니다.

### 3️⃣ 라이브러리 (실행 시점 **실측** 버전)

| 라이브러리 | 버전 | 무엇에 쓰나 |
|---|---|---|
| `torch` | 2.12.1 | Sionna PHY 백엔드 — ⚠ Sionna 2.0 은 TensorFlow 가 아니라 **PyTorch** |
| `numpy` | 2.5.0 | 수치 계산 전반 |
| `matplotlib` | 3.11.0 | 도표 |

### 4️⃣ 어디서 돌렸나

- **Python** 3.12.13 · Linux 5.15.0-136-generic
- **GPU** — `src/gpu.py` 가 **여유 메모리를 보고 자동 선택**합니다 (하드코딩 없음):
  - 0, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 1, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 2, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 3, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
- `CUDA_VISIBLE_DEVICES` = (고정 안 함 — src/gpu.py 가 여유 메모리 보고 자동 선택)

- **계산 비용**: verify_* 스크립트 각 GPU 1장 수 분. 그림은 이미 산출된 것을 재사용.

### 5️⃣ 어떻게 다시 돌리나 (재현)

```bash
cd /home/yunjung/workspace/sionna2
~/.venvs/py312/bin/python benchmark/verify_eca.py
~/.venvs/py312/bin/python benchmark/verify_ambiguity.py
~/.venvs/py312/bin/python benchmark/verify_linkbudget.py
~/.venvs/py312/bin/python src/make_notebook11.py     # report11.ipynb 재생성
```

### 6️⃣ 본문 숫자는 어디서 오나

이 노트북의 **숫자는 손으로 적지 않았습니다.** 측정 스크립트가 JSON 을 남기고, 노트북 생성기(`src/make_notebook*.py`)가 그 JSON 을 읽어 본문에 주입합니다. → **그림과 글이 어긋날 수 없습니다.** 숫자가 이상하면 JSON 을 보세요.

### 7️⃣ 무엇이 산출되나

| 산출물 | 무엇 |
|---|---|
| `outputs/verify_eca.json` | ECA 노치·블라인드 속도 |
| `outputs/verify_ambiguity.json` | 모호함수·거리/속도 분해능 |
| `outputs/verify_linkbudget.json` | 레이더방정식·잡음·SCR·Pd |
| `outputs/figures/verify_eca.png` | ECA 저속 맹점 그림 (재사용) |
| `outputs/figures/verify_ambiguity_af.png` | 모호함수 거리-도플러 지도 (재사용) |

### 8️⃣ ⚠️ 믿으면 안 되는 것 (신뢰 경계)

> 정직함이 이 프로젝트의 규칙입니다. **아래는 이 리포트가 보장하지 않는 것들입니다.**

- **블라인드 속도는 적분시간(CPI)에 달렸다.** 여기 값은 운용 적분시간 기준이다. 더 오래 적분하면 노치가 좁아져 더 느린 표적까지 보이지만 관측이 그만큼 느려진다(맞바꿈).
- **분해능은 신호가 상시 쓰는 기준신호의 대역폭이 정한다.** 5G 의 상시 신호(SSB)는 대역이 7.2 MHz 뿐이라 거리 분해능이 39 m 로 거칠다.
- **여기서 다루는 것은 탐지(거리+속도 셀에서 '있다/없다')까지다.** 표적의 위치·궤적을 잇는 추적은 이 리포트의 범위 밖이다.

### 9️⃣ 앞뒤 리포트

| 리포트 | 관계 |
|---|---|
| report10 (앞) | 오경보율 — 검출기가 '헛것'을 얼마나 자주 보나(문턱 눈금) |
| [report12](report12.ipynb) (다음) | 결과편 — 다중 수신기(1→4) 디텍션 + 9모드 벤치마크. §5 의 '수신기 2개면 위치가 풀린다' 가 거기서 '수신기를 늘리면 감도도 오른다(+10log10 N dB)' 로 이어진다 |

<details><summary><b>🔤 용어집 — 모르는 말이 나오면 여기</b> (클릭)</summary>

| 용어 | 뜻 |
|---|---|
| **도플러 f_d** | 표적이 움직여 생기는 수신 주파수의 이동. 정지물(클러터)과 움직이는 표적을 가르는 축. 느린 표적일수록 f_d 가 0 에 가까워 클러터와 구별이 어렵다 |
| **ECA (직접파 제거)** | 송신기에서 수신기로 곧장 새어 든 강한 직접파를, 지연된 기준신호가 만드는 부분공간에 사영해 빼는 전처리. **정지 성분(도플러 0)을 통째로 지운다** — 이 성질이 곧 저속 맹점의 뿌리다 |
| **블라인드 속도** | ECA 노치에 먹혀 되돌아온 신호가 3 dB 이상 깎이는 최소 시선속도. 이보다 느리면 표적이 **원리적으로** 잘 안 보인다 |
| **모호함수 / CAF** | 교차모호함수(Cross-Ambiguity Function). 기준신호와 수신신호를 **거리(지연)와 도플러로 동시에** 상관시켜 만든 2차원 지도. 봉우리 폭이 분해능, 곁봉우리가 가짜표적 |
| **거리 분해능 ΔR_b** | 두 표적을 거리축에서 갈라 볼 수 있는 최소 간격. 바이스태틱에서 ΔR_b = c/B (B = 기준신호 대역폭). 대역이 넓을수록 촘촘히 가른다 |
| **sinc / 0.886** | 직사각 창의 스펙트럼 모양(sin πx / πx). 그 −3 dB 폭은 이론값의 **0.886배**다. 측정/이론 비가 이 값 근처면 분해능 눈금이 맞은 것 |
| **링크버짓** | 표적 밝기(σ)·거리·잡음을 곱하고 나눠 되돌아온 신호대잡음비(SNR)를 예측하는 계산. = 레이더 방정식 |
| **RCS (σ)** | 레이더 되비침 밝기. σ 는 그 밝기를 넓이 단위(m²)로 적은 값. 표적이 밝아야(σ 가 커야) 잡힌다. Sionna 기본 solver 는 이 σ 를 못 내므로 **SBR** 로 따로 계산해 넣는다 |
| **SBR** | 표적을 광선으로 조준해 맞고, 맞은 면이 레이더로 되쏘는 양을 물리광학(PO)으로 계산·합산해 σ 를 구하는 방법 |
| **SCR** | 신호대클러터비(Signal-to-Clutter Ratio). 거리-도플러 지도에서 표적 봉우리 ÷ 주변 바닥. 검출 난이도의 실측 지표 |
| **Pd (검출확률)** | 표적이 있을 때 검출기가 실제로 발화할 확률. SCR 이 올라갈수록 0 에서 1 로 전이한다 |
| **CFAR** | 주변 잡음을 보고 문턱을 스스로 정하는 검출기(Constant False Alarm Rate) |
| **CPI / 적분시간** | 한 번 관측하며 신호를 모아 쌓는 시간(Coherent Processing Interval). 길수록 도플러를 촘촘히 보지만 관측이 느려진다 |
| **semi-anechoic** | 벽·천장은 전파를 흡수하고 바닥만 반사하는 반무향 챔버 — 우리 실험 환경 |

</details>

---


---
# §1. Sionna 의 공백과 ECA 의 저속 맹점 — 블라인드 속도

**공백부터.** 검출은 (1) 직접파·정지 클러터 제거(ECA) → (2) 거리-도플러 상관(CAF) → (3) 문턱 검출(CFAR)의 **표준 처리 사슬**로 이뤄진다. 그러나 스톡 Sionna 는 파형·전파(경로·지연·도플러)만 줄 뿐 **이 레이더 신호처리를 아예 갖고 있지 않다.** 그래서 여기서는 표준 사슬을 그대로 따르되, 오픈소스 **pyAPRiL**(GPLv3, BME Radarlab)이 제공하는 그 ECA/CAF/CFAR 를 쓴다 — 실 하드웨어 선행이 쓴 체인과 같다(5G NR 패시브 레이더 Wypich·Zielinski, Sensors 2026, ECA⁺→CA-CFAR; LTE450 드론 패시브 레이더 Demissie, IET RSN 2025). 이 절은 그 첫 단계 ECA 가 **느린 표적에 만드는 원리적 맹점**을 본다.

ECA 는 도플러가 0 인 성분(움직이지 않는 벽·바닥·직접파)을 지우도록 설계돼 있다. 그런데 표적이 느리면 그 표적의 도플러도 0 에 가까워져 **ECA 가 표적을 정지 클러터로 착각해 함께 지운다.** 지워지는 정도는 표적의 도플러가 '도플러 한 칸(=1/적분시간)'에서 얼마나 떨어져 있느냐로 정해진다.

**근거 — 노치의 모양.** ECA 가 표적을 깎는 양은 이론적으로 $1-\mathrm{sinc}^2(f_d\,T)$ 라는 골짜기(노치) 모양을 따른다(T = 적분시간). 측정한 손실 곡선은 이 이론과 최대 **0.002 dB** 밖에 차이 나지 않는다 — 노치 폭은 정확히 **도플러 한 칸**이다. 아래 표는 5G 신호에서 표적 도플러가 도플러 한 칸의 몇 배일 때 얼마나 깎이는지다(이론과 나란히).

| 표적 도플러 ÷ 한 칸 | 측정 에너지 손실 | 이론 $1-\mathrm{sinc}^2$ |
|---|---|---|
| 0.1 | -14.89 dB | -14.89 dB |
| 0.2 | -9.04 dB | -9.04 dB |
| 0.3 | -5.80 dB | -5.80 dB |
| 0.5 | -2.26 dB | -2.26 dB |
| 0.8 | -0.24 dB | -0.24 dB |
| 1.0 | +0.00 dB | +0.00 dB |

**블라인드 속도.** 되돌아온 에코 에너지가 −3 dB(절반)로 깎이는 지점을 시선속도로 환산하면, 운용 적분시간에서 이렇게 나온다.

| 신호 | 적분시간 | 도플러 한 칸 | 블라인드 속도(−3 dB) |
|---|---|---|---|
| 5G NR 100MHz | 24 ms | 41.7 Hz | **0.88 m/s** |
| WiFi 80MHz | 48 ms | 20.8 Hz | **0.29 m/s** |
| LTE 20MHz | 48 ms | 20.8 Hz | **0.83 m/s** |

즉 초속 **0.29~0.88 m/s** 보다 느린 표적은 −3 dB 이상 깎여 사실상 놓친다. **제자리에 떠 있는(호버) 드론은 시선속도가 0 에 가까워 원리적으로 이 골짜기 안에 갇힌다.** 적분시간을 96 ms 로 더 늘리면 노치가 좁아져 **0.15 m/s** 까지 내려가지만, 그만큼 한 번 보는 데 오래 걸린다(관측을 느리게 만드는 맞바꿈).

아래 그림 (c) 는 모든 신호·적분시간의 손실 곡선이 하나의 $1-\mathrm{sinc}^2$ 곡선으로 겹침을, (d) 는 신호별 저속 사각지대(이 속도 아래로는 표적을 먹는다)를 보여 준다.

![ECA blind-speed notch and slow-drone blind zone](outputs/figures/verify_eca.png)

---
# §2. 모호함수 — 신호가 표적을 얼마나 또렷이 가르나

**모호함수(CAF)** 는 기준신호를 거리(지연)와 속도(도플러)로 동시에 훑어 만든 2차원 지도다 — 표준 검출 사슬의 2단계이자 pyAPRiL 이 제공하는 그 상관 지도다. 한가운데 봉우리가 뾰족할수록 두 표적을 잘 가르고(분해능), 봉우리 옆에 솟은 곁봉우리는 없는 표적을 있는 것처럼 보이게 하는 가짜표적이다. 여기서는 각 신호가 그 지도에서 두 표적을 얼마나 촘촘히 가르는지(거리 분해능)를 이론과 대조한다.

**근거 — 거리 분해능.** 바이스태틱 거리 분해능의 이론값은 $\Delta R_b = c / B$ (B = 기준신호 대역폭; 모노스태틱 등가는 $c/2B$). 직사각 창이면 실제 −3 dB 봉우리 폭은 이론의 **0.886배**(sinc 폭)가 나와야 정상이다. 측정 결과:

| 신호(기준) | 기준 대역폭 | 측정 ΔR_b | 이론 c/B | 측정/이론 | 곁봉우리(챔버) |
|---|---|---|---|---|---|
| WiFi 80 MHz · VHT-LTF | 76.6 MHz | 3.50 m | 3.92 m | **0.895** | -23.4 dB |
| LTE 20 MHz · CRS | 18.0 MHz | 15.33 m | 16.67 m | **0.920** | -15.0 dB |
| 5G NR 100 MHz (상시 SSB) · SSB | 7.2 MHz | 39.21 m | 41.64 m | **0.942** | -18.3 dB |
| 5G NR 100 MHz (측위 PRS) · NR-PRS | 98.3 MHz | 2.77 m | 3.05 m | **0.910** | -15.7 dB |

측정/이론 비가 전부 **0.89~0.94**(sinc 의 0.886 근처)로 거리 눈금이 맞았다. **대역이 넓을수록 촘촘히 가른다** — 5G 의 측위용 신호(NR-PRS, 98 MHz)는 2.8 m 까지 가르지만, 5G 가 **상시** 내보내는 신호(SSB)는 대역이 7.2 MHz 뿐이라 39 m 로 거칠다. 즉 5G 로 상시 탐지하려면 거리 분해능을 크게 손해 본다 — 이 SSB 상시모드의 한계는 5G NR SSB 패시브 바이스태틱 드론 검출을 거리·속도 CRB 로 분석한 선행(Jopanya·Osorio, SPAWC 2025)이 겨냥한 바로 그 조명원이다.

**곁봉우리(가짜표적)는 챔버 밖에 있다.** OFDM 신호의 반복 구조는 곁봉우리를 만들지만, 그 봉우리들은 수백 m~km 거리에 찍혀 우리 챔버 관측창(바이스태틱 거리 ±60 m 안, 도플러 ±260 Hz) 밖이다. 창 안에서 곁봉우리는 표적보다 최소 15 dB 낮아 무해하다.

**이 모호함수는 실제 검출에 쓰는 거리-도플러 지도와 같은 것이다** — 둘을 맞대 보면 최대 **0.14 dB** 밖에 차이 나지 않는다. 아래 그림은 신호별 모호함수 지도(위)와 거리축 단면(아래)이다.

![Ambiguity function |chi(tau,fd)| per waveform](outputs/figures/verify_ambiguity_af.png)

![Zero-Doppler range cut — mainlobe width = range resolution](outputs/figures/verify_ambiguity_range.png)

![.](outputs/renders/anim/ambiguity_nr.gif)

<sub>5G 신호의 모호함수(거리-도플러) — 가운데 봉우리가 좁고 날카로울수록 표적을 또렷이 가른다.</sub>

![.](outputs/renders/anim/ambiguity_lte.gif)

<sub>LTE 신호의 모호함수 — 대역이 좁아 거리축으로 더 퍼진다(분해능이 거칠다).</sub>

---
# §3. 링크버짓 — 밝기·거리·잡음이 신호대잡음비를 만든다

링크버짓(레이더 방정식)은 **표적 밝기(σ) × 거리 감쇠 ÷ 잡음** 으로 되돌아온 신호대잡음비(SNR)를 예측한다. 이 절은 그 계산을 독립적인 세 방법으로 세워 서로, 그리고 이론과 맞는지 확인한다.

여기서 표적 밝기 σ 는 Sionna RT 가 주지 못한다 — 창설 논문(arXiv:2303.11103)이 문서화하듯 스톡 PathSolver 는 **경로별 복소이득만** 반환하고 표적 표면 위 산란적분 단계가 없다. 그래서 Sionna 로 센싱하는 선행은 σ 를 **밖에서 계산해 채널에 주입**한다 — LAMBDA(arXiv:2607.03826)는 Sionna 전파에 상용 EM(CADFEKO) UAV RCS 를, Temporal-GNN(arXiv:2604.08306)은 점산란체 RCS 를 결합한다. 선행이 쓰는 세 갈래(상용 full-wave / 자작 SBR+PO / 점산란체) 중 여기서는 **자작 SBR+PO** 갈래를 따른다 — GPU SBR+PO(BVH SBR+PO, arXiv:2604.09243)가 쓰는 것과 같은 방법(광선으로 조명면을 찾아 그 위에서 물리광학 적분)이며, 검증은 이론(평판·구)과 **실측 문헌 드론 RCS 앵커**(report08)로 한다.

**근거 — 세 계산이 일치한다.** 같은 SNR 을 (a) 닫힌형 공식, (b) 단계별 사슬 계산, (c) dB 산술 세 방법으로 구했더니 서로 최대 **3e-14 dB** 밖에 차이 나지 않는다(사실상 완전 일치). 신호를 모아 쌓는 **처리이득**도 이론 대비 **0.096 dB**, **잡음바닥**은 대역폭 보정을 넣으면 이론과 **1e-15 dB** 로 맞는다. 계산 사슬이 통째로 검증됐다.

| 신호 | 처리이득(이론) | 처리이득(측정) | 잡음바닥 오차 |
|---|---|---|---|
| WiFi 80MHz | 1.74 dB | 1.77 dB | -1e-15 dB |
| LTE 20MHz | 15.88 dB | 15.79 dB | +0e+00 dB |
| 5G 100MHz | 19.21 dB | 19.20 dB | +0e+00 dB |

---
# §4. 우리가 쓴 방식과 검증 — 디텍션이 실제로 작동한다

앞의 §1~§3 은 '한계'와 '눈금'을 다뤘다. 이제 핵심 질문: **그래서 잡히긴 하나?** 표적 봉우리가 주변 바닥보다 충분히 높으면(SCR 이 크면) 검출기가 발화한다. SCR 이 낮으면 놓치고, 어느 문턱을 넘으면 확실히 잡는다. 그 전이를 검출확률 곡선으로 본다.

**검출체인은 검증된 표준이다.** ECA→CAF→CFAR 사슬 자체는 오픈소스 pyAPRiL(GPLv3)이 제공하는 그 체인이며, 그대로 돌려 NR·WiFi·LTE **3/3 모드 모두** 표적을 정답 거리빈(오차 0 빈)에 검출함을 확인했다(outputs/verify_pyapril.json). 아래 **대량 몬테카를로 Pd 곡선은 그 같은 표준 체인을 GPU 로 대량 반복해** 얻은 것이다 — 검증된 체인이고, 대량 통계만 GPU 로 낸다.

**근거 — 드론이 모두 잡힌다.** SBR 로 계산한 σ 를 넣으니 5 대 드론 × 3 신호 = 15 조합의 측정 SCR 이 **14.8~55.3 dB** 로 나오고, **전부 Pd = 1.0(모두 검출)** 이다. 링크버짓으로 예측한 SCR 과 실제 지도에서 잰 SCR 의 차이는 평균 **-1.9 dB**(±1.3) 로, 이 차이는 오류가 아니라 창 가중 등 **예측 가능한 처리 손실**이다.

| 드론 | 표적 밝기 σ | 예측 SCR | 측정 SCR | 검출 |
|---|---|---|---|---|
| mini5pro | -27.5 dBsm | 15.7 dB | 14.8 dB | ✅ Pd=1.0 |
| mavic4pro | -16.0 dBsm | 27.1 dB | 26.0 dB | ✅ Pd=1.0 |
| matrice4e | -21.5 dBsm | 21.7 dB | 20.7 dB | ✅ Pd=1.0 |
| s1000plus | -14.2 dBsm | 29.0 dB | 27.9 dB | ✅ Pd=1.0 |
| phantom4 | -19.9 dBsm | 23.3 dB | 22.3 dB | ✅ Pd=1.0 |

*(WiFi 조명 예시 — LTE·5G 도 같은 경향. 작은 드론일수록 σ 가 어둡지만 그래도 문턱 위에 있다.)*

<sub>여기 **σ 는 이 챔버의 특정 바이스태틱 기하**(TX·RX·표적 배치, 단일 반송파, 8° 창 국소평균)에서 잰 값이다 — 그래서 report08 §2 의 **360° 방위평균·밴드평균 σ 와 값이 다르다**(RCS 는 자세·기하에 따라 수 dB 출렁인다). 링크버짓에는 이 기하에서의 실제 밝기를 그대로 넣는다.</sub>

**검출은 SCR 5~15 dB 구간에서 전이한다.** SCR 을 바꿔 가며 검출확률을 재면(5G 조명, 운용 오경보율 $10^{-4}$ 기준) 아래처럼 0 에서 1 로 올라선다.

| SCR | 검출확률 Pd |
|---|---|
| +3.4 dB | 0.001 |
| +3.5 dB | 0.002 |
| +3.8 dB | 0.000 |
| +5.1 dB | 0.020 |
| +9.2 dB | 0.447 |
| +14.8 dB | 1.000 |

SCR 5 dB 근처에서는 거의 못 잡다가 15 dB 에 이르면 확실히 잡는다. **우리 드론들의 운용 SCR 은 모두 14 dB 이상**(최저 14.8 dB)으로 이 전이 구간의 위쪽에 있으니 확실히 검출된다.

> 🔑 **이 절이 이 리포트의 결론이다.** 표적을 밝게 만드는 σ 는 탐지에 반드시 필요하고(밝아야 봉우리가 선다), 그 σ 를 Sionna RT 로는 못 내므로 **선행이 하는 대로 자작 SBR+PO 로 계산해 채널에 주입한다.** 검출은 pyAPRiL 로 검증된 표준 체인으로 하고 대량 Pd 통계만 GPU 로 낸다. 그렇게 하면 **실제로 탐지가 된다** — 위 표가 그 증거다.

---
# §5. 관측가능성 — '있다/없다' 다음에 오는 '어디에 있나'

지금까지는 **탐지**(거리·속도 칸에 표적이 있나)를 다뤘다. 이 절은 그 다음 질문, **위치를 알 수 있나**를 본다. 답은 분명하다: **송·수신기 한 쌍으로는 원리적으로 불가능**하다.

### 5.1 정확도 ≠ 분해능 — 흔한 오해부터

**분해능(resolution)** 은 *두 표적을 가를 수 있나*(§2 의 거리 빈 폭)이고, **정확도(accuracy)** 는 *표적 하나의 위치를 얼마나 정밀히 찍나*(CRLB, 신호대잡음비에 좌우)입니다. **둘은 완전히 다른 축**인데 빈 폭을 정확도로 오해하기 쉽습니다. 이 거리·속도 정확도의 하한(CRB)은 선행에서 다뤄졌습니다 — 5G SSB 패시브 바이스태틱 드론 검출의 거리·속도 CRB 를 유도한 Jopanya·Osorio(SPAWC 2025)가 그 예입니다. 실측하면 정확도가 분해능보다 **수십~수백 배 미세**합니다:

| 신호 | 유효 셀 크기 (max[빈그리드, 주엽폭]) | 정확도 σ_R_b (CRLB) | 정확도가 더 미세한 배율 | SCR |
|---|---|---|---|---|
| WiFi80 G1 (VHT-LTF) | 3.75 m | 0.036 m | **104배** | 32.6 dB |
| LTE20 G1 (CRS) | 12.86 m | 0.014 m | **892배** | 53.4 dB |
| 5G100 G1 (SSB) | 39.20 m | 1.490 m | **26배** | 21.1 dB |
| 5G100 G3 (PRS) | 2.44 m | 0.008 m | **289배** | 43.2 dB |

<sub>표적이 **하나뿐**이면 WiFi80 는 빈 폭이 3.7 m 라도 그 표적의 거리를 **3.6 cm** 급으로 찍을 수 있습니다. 빈 폭은 '두 표적을 가르는 능력'이지 '한 표적을 재는 정밀도'가 아닙니다. (※ FFT 제로패딩은 정확도를 CRLB 까지 끌어올리지만 **분해능은 1도 올리지 않습니다** — 빈을 잘게 쪼갤 뿐 신호 대역이 넓어지는 게 아니니까요. 위 표의 '유효 셀 크기' 는 max(FFT 빈 그리드, 주엽폭) 이라 §2 의 −3 dB 주엽폭 ΔR_b 와는 정의가 다릅니다.)</sub>

### 5.2 그런데 위치는 못 찍는다 — 송·수신기 한 쌍의 원리적 한계

한 순간에 우리가 재는 것은 **바이스태틱 거리 R_b 와 도플러 f_d**, 딱 **두 개**입니다. 그런데 위치는 **3차원**입니다. 미지수 3개를 방정식 2개로 풀 수 없습니다 — 측정으로 확인한 판정: **NOT OBSERVABLE with one TX-RX pair** (스냅샷 FIM 랭크 **2 / 상태차원 3**).

> **비유 —** 종이 위에 그린 타원을 생각하세요. 압정 두 개(송신기·수신기)에 실을 걸고 연필을 팽팽히 당겨 그리면 타원이 나옵니다. R_b 를 안다는 건 '표적이 이 타원 위 어딘가'라는 뜻이지 '타원 위 어디'인지가 아닙니다. 3D 에선 타원이 **회전타원체(껍질)** 가 되고, 표적은 그 껍질 어딘가에 있습니다.

게다가 이 모호함은 근사가 아니라 **엄밀한 대칭**입니다: **rotation about the TX-RX baseline (verified to machine precision)** — 표적을 송수신기를 잇는 축 둘레로 회전시켜도 R_b 와 f_d 가 **바뀌지 않습니다**. 수치로 확인하면 회전시켜도 변화가 ΔR_b < 1e-14 m, Δf_d < 4e-14 Hz — **기계 정밀도(부동소수점 오차) 수준**, 즉 정확히 0 입니다. 시간을 두고 여러 번 봐도(관측 그램행렬) 이 축 방향은 끝내 채워지지 않습니다.

![observability shell](outputs/figures/report4_obs_shell.png)

<sub>바이스태틱 거리 R_b 하나가 정하는 것은 '점'이 아니라 **회전타원체 껍질**이다. 도플러를 더해도 껍질 위의 곡선까지만 좁혀진다.</sub>

![baseline-axis rotation keeps R_b invariant](outputs/renders/anim/obs_baseline_ring.gif)

> **움직이는 그림:** 표적을 **TX-RX 기저선 축 둘레로 회전**시키면 위치는 계속 바뀌는데 바이스태틱 거리 $R_b=R_1+R_2-L$ 은 **기계정밀도(≈1.4×10⁻¹⁴ m)까지 그대로**다. 즉 한 쌍의 관측만으로는 이 회전 방향을 원리적으로 구별할 수 없다 — 위 '껍질'이 왜 점으로 좁혀지지 않는지의 직접 증거.

![observability CRLB](outputs/figures/report4_obs_crlb.png)

<sub>위치 정확도(CRLB). 단일 송수신 쌍에서는 baseline 축 방향으로 오차가 **발산**한다 — 그 방향으로는 정보가 0 이기 때문.</sub>

### 5.3 처방 — 수신기를 늘리거나, 각도를 재거나

정보가 없는 방향을 채우려면 **다른 각도에서 한 번 더 보는 수밖에** 없습니다. 세 처방을 같은 조건에서 재보면:

| 구성 | 실효 랭크 | 조건수 | **위치 RMS 오차** |
|---|---|---|---|
| 수신기 1개 (기준) | 3 / 6 | 7e+19 | **66.7 m** (사실상 못 씀) |
| **수신기 2개** | 6 / 6 | 8e+05 | **0.219 m** |
| 수신기 1개 + 각도(AoA) 1° | 6 / 6 | 1e+05 | **0.121 m** |
| 수신기 1개 + 각도(AoA) 5° | 6 / 6 | 3e+06 | **0.603 m** |

**읽는 법.** 수신기 하나면 랭크가 모자라(조건수 7e+19 — 사실상 특이) 위치 오차가 **67 m** 로 터집니다. **두 번째 수신기를 놓거나**(→ 22 cm) **각도를 1° 정밀도로 재면**(→ 12 cm) 랭크가 6/6 으로 차고 위치가 풀립니다.

> **그래서 [report12](report12.ipynb) 의 다중 수신기 배열이 중요합니다.** 거기서는 수신기를 늘려 **탐지 감도**(+10·log10 N dB)를 얻었는데, 같은 배열이 여기서는 **위치 관측가능성**까지 열어줍니다 — 탐지에는 수신기 1개로 충분하지만, **추적(위치·궤적)에는 2개 이상 또는 각도 측정이 필수**입니다. 이것이 이 프로젝트가 탐지를 먼저 하고 추적을 다음 일로 미룬 이유입니다. 추적 자체는 직접 짜지 않고 오픈소스 추적 프레임워크(Stone Soup, MIT)에 바이스태틱 측정모델만 얹어 붙일 계획이며, 셀룰러 조명·다중 수신기로 패시브 드론을 추적한 실측 선행(Fan Liu 외, 도플러 다중스태틱, arXiv:2509.25732)이 이미 있습니다.

![observability gramian](outputs/figures/report4_obs_gramian.png)

<sub>시간을 두고 관측을 쌓아도(관측 그램행렬) baseline 축 방향의 고윳값은 채워지지 않는다 — 시간이 아니라 **기하**가 부족한 것이라, 오래 본다고 풀리지 않는다.</sub>

---
## 맺음

검출에는 두 종류의 문턱이 있다. **속도 쪽 문턱**: 직접파 제거(ECA)는 정지 성분을 지우므로 초속 약 0.29~0.88 m/s 보다 느린 표적, 특히 호버 드론을 원리적으로 함께 지운다. **분해능 쪽 문턱**: 두 표적을 가르는 거리 해상도는 상시 신호의 대역폭이 정하며, 측정치는 교과서 값의 0.89~0.94배로 정확하다. 그 사이에서, 밝기·거리·잡음을 잇는 링크버짓은 이론과 소수점까지 맞고, **SBR 로 계산한 표적 밝기(σ)를 넣으면 드론이 실제로 검출된다**(SCR 15~55 dB, Pd=1.0).

**요약하면, 각 조각은 표준·선행·라이브러리로 뒷받침된다:** 검출 사슬(ECA→CAF→CFAR)은 pyAPRiL 로 검증한 패시브 레이더 표준이고, 표적 밝기 σ 는 선행이 쓰는 자작 SBR+PO 방식으로 계산해 주입하며, 파형·전파는 Sionna PHY·RT 다. **다음 단계는 실증(USRP X410, OpenISAC+GNU Radio)과 추적(Stone Soup, 거리+속도)이다.**